# Ethiopian Agricultural AI — FINAL Colab Notebook

This version is built from the notebook you attached, but the unstable parts were replaced.

## Main engineering decision
Your latest error is caused by `bitsandbytes` + CUDA 13 in Colab, not by Qwen itself. This final notebook therefore **does not use bitsandbytes, TRL, SFTTrainer, or paged_adamw_8bit**.

Instead it uses:

- Qwen2.5 base model loaded directly in FP16
- PEFT LoRA adapters
- Hugging Face `Trainer`
- `adamw_torch` optimizer
- assistant-only label masking
- T4-safe batch size and sequence length

This avoids the exact failure you saw: `libnvJitLink.so.13` / `Bnb4bitQuantize`.

## Run order
Run cells from top to bottom. After Section A installs packages, restart the runtime once, then continue from Section B.

## SECTION A — Install stable packages

Run this once, then restart runtime. Do **not** install `bitsandbytes`, `trl`, `triton`, or `torchvision` in this notebook.

In [ ]:
# SECTION A — Stable installation for Colab T4/A100
# Run once, then Runtime -> Restart session.

import sys, subprocess, os

packages = [
    "transformers==4.46.3",
    "accelerate==1.1.1",
    "peft==0.13.2",
    "datasets==3.1.0",
    "huggingface_hub>=0.26.0",
    "sentencepiece==0.2.0",
    "protobuf>=4.25.0",
    "safetensors>=0.4.5",
    "pandas==2.2.2",
    "numpy<2.1",
    "scikit-learn==1.5.2",
    "matplotlib",
    "tqdm",
    "gspread>=6.0.0",
    "google-auth>=2.0",
    "openpyxl",
]

print("Installing stable packages. Do not interrupt...")
cmd = [sys.executable, "-m", "pip", "install", "-q", "--upgrade"] + packages
result = subprocess.run(cmd, text=True, capture_output=True)

if result.returncode != 0:
    print(result.stderr[-4000:])
    raise RuntimeError("Package installation failed. See error above.")

print("✅ Packages installed.")
print("IMPORTANT: Runtime -> Restart session, then continue from SECTION B.")

Installing stable packages. Do not interrupt...


KeyboardInterrupt: 

## SECTION B — Environment check and Hugging Face login

This cell blocks unsafe states early. It also avoids hardcoding your Hugging Face token.

In [ ]:
# SECTION B — Environment check + Hugging Face login

import os, sys, gc, getpass, importlib.metadata as im
from pathlib import Path
from datetime import datetime

import torch
from huggingface_hub import login, whoami, HfApi

print("Python:", sys.version.split()[0])
print("Torch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)

if not torch.cuda.is_available():
    raise RuntimeError("❌ GPU is not available. In Colab: Runtime -> Change runtime type -> T4 GPU.")

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print("GPU:", gpu_name)
print(f"VRAM: {vram_gb:.1f} GB")

# IMPORTANT: use FP16 on T4. T4 is not a real BF16 training GPU.
USE_BF16 = False
USE_FP16 = True
DTYPE = torch.float16
print("Training dtype: FP16")

for pkg in ["transformers", "accelerate", "peft", "datasets"]:
    print(f"{pkg}:", im.version(pkg))

# Fail early if broken packages from older notebook are installed and imported accidentally.
for bad_pkg in ["bitsandbytes", "trl"]:
    try:
        print(f"⚠️ {bad_pkg} is installed, but this notebook will NOT use it:", im.version(bad_pkg))
    except im.PackageNotFoundError:
        print(f"✅ {bad_pkg} not installed / not needed")

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = None

if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Paste Hugging Face READ token (input hidden): ").strip()

if not HF_TOKEN.startswith("hf_"):
    raise ValueError("❌ Invalid Hugging Face token. It should start with hf_")

login(token=HF_TOKEN, add_to_git_credential=False)
info = whoami(token=HF_TOKEN)
print("✅ Hugging Face login successful as:", info.get("name", "unknown"))

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
api = HfApi(token=HF_TOKEN)
api.model_info(MODEL_ID)
print("✅ Qwen access OK:", MODEL_ID)

## SECTION C — Google Drive and Google Sheet loading

Edit `SHEET_ID`, `SHEET_NAME`, `USER_COL`, and `ASSISTANT_COL` only if your sheet changed.

In [ ]:
# SECTION C — Google Drive + Google Sheets loading

import os, json, re, warnings
from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np
warnings.filterwarnings("ignore")

# EDIT ONLY THESE VALUES IF NEEDED
SHEET_ID = "1-e2UaXMuZSFQkXSbFdDL4Go-UN1rgtZWQ8KdZ61rioE"
SHEET_NAME = "amharic_agriculture_finetune_expanded"
USER_COL = "user"
ASSISTANT_COL = "assistant"

from google.colab import drive, auth
import gspread
from google.auth import default

drive.mount("/content/drive", force_remount=False)
auth.authenticate_user()

TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M")
DRIVE_BASE = Path("/content/drive/MyDrive/amharic_agri_ai")
RUN_DIR = DRIVE_BASE / f"final_lora_no_bnb_{TIMESTAMP}"
ADAPTER_DIR = RUN_DIR / "lora_adapter"
REPORT_DIR = RUN_DIR / "reports"
CHECKPOINT_DIR = RUN_DIR / "checkpoints"
for p in [RUN_DIR, ADAPTER_DIR, REPORT_DIR, CHECKPOINT_DIR]:
    p.mkdir(parents=True, exist_ok=True)
print("✅ Output folder:", RUN_DIR)

creds, _ = default()
gc_client = gspread.authorize(creds)
worksheet = gc_client.open_by_key(SHEET_ID).worksheet(SHEET_NAME)
records = worksheet.get_all_records()
df_raw = pd.DataFrame(records)

if USER_COL not in df_raw.columns or ASSISTANT_COL not in df_raw.columns:
    raise ValueError(f"Required columns missing. Found columns: {df_raw.columns.tolist()}")

print(f"✅ Loaded sheet: {len(df_raw):,} rows")
print("Columns:", df_raw.columns.tolist())

## SECTION D — Data cleaning

This removes empty rows, duplicates, very short rows, and badly mixed-language rows. The thresholds are conservative so useful Amharic agriculture rows are kept.

In [ ]:
# SECTION D — Data cleaning

import re, pandas as pd, numpy as np

EMPTY = {"", "nan", "none", "null", "n/a", "na"}

def clean_text(x):
    x = "" if x is None else str(x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

def ethiopic_ratio(text):
    text = clean_text(text)
    if not text:
        return 0.0
    # Count real letters/scripts only, not spaces/punctuation.
    script_chars = re.findall(r"[\u1200-\u137F\u0041-\u007A\u0600-\u06FF\u4E00-\u9FFF]", text)
    eth_chars = re.findall(r"[\u1200-\u137F]", text)
    return len(eth_chars) / max(len(script_chars), 1)

df = df_raw[[USER_COL, ASSISTANT_COL]].copy()
df[USER_COL] = df[USER_COL].map(clean_text)
df[ASSISTANT_COL] = df[ASSISTANT_COL].map(clean_text)

before = len(df)
df = df[~df[USER_COL].str.lower().isin(EMPTY)]
df = df[~df[ASSISTANT_COL].str.lower().isin(EMPTY)]
df = df.dropna(subset=[USER_COL, ASSISTANT_COL])
df = df.drop_duplicates(subset=[USER_COL, ASSISTANT_COL])

# Basic length filters: avoid garbage and extreme rows that waste T4 memory.
df["user_len"] = df[USER_COL].str.len()
df["assistant_len"] = df[ASSISTANT_COL].str.len()
df["answer_amharic_ratio"] = df[ASSISTANT_COL].map(ethiopic_ratio)

df = df[df["user_len"].between(3, 1200)]
df = df[df["assistant_len"].between(10, 2500)]
# Keep rows where assistant answer is mostly Amharic. Use 0.55 to allow numbers/scientific terms.
df = df[df["answer_amharic_ratio"] >= 0.55]

df = df[[USER_COL, ASSISTANT_COL]].reset_index(drop=True)

print(f"Raw rows: {before:,}")
print(f"Clean rows: {len(df):,}")
print(f"Removed: {before - len(df):,}")

if len(df) < 100:
    raise RuntimeError("❌ Too few clean rows. Lower the Amharic ratio threshold only if your data is intentionally mixed-language.")

print("\nSample rows:")
for i in range(min(3, len(df))):
    print("- USER:", df.loc[i, USER_COL][:160])
    print("  ASSISTANT:", df.loc[i, ASSISTANT_COL][:180])

# Save cleaned copy for audit.
df.to_csv(REPORT_DIR / "cleaned_training_data_preview.csv", index=False)
print("✅ Data cleaning complete")

Raw rows: 52,332
Clean rows: 52,332
Removed: 0

Sample rows:
- USER: የተባይ አስተዳደር ምንድን ነው?
  ASSISTANT: የተባይ አስተዳደር ተክሎችን ከጎጂ ተባዮች በተመጣጣኝ መንገድ መጠበቅ ለማሳካት የሚረዳ ስርዓት ነው። በአካባቢ ሁኔታ መሠረት አተገባበሩ ሊለያይ ይችላል። ዋና ዓላማው ተክሎችን ከተባይ፣ ከበሽታ እና ከአረም ጫና መጠበቅ ነው። በተጨማሪም በአካባቢ ሁኔታ መሠረት መልኩ መለያየት ይኖርበታል
- USER: የተባይ አስተዳደር ምን ማለት ነው?
  ASSISTANT: የተባይ አስተዳደር ማለት ተክሎችን ከጎጂ ተባዮች በተመጣጣኝ መንገድ መጠበቅ ለማጠናከር የሚያገለግል ስርዓት ነው። በአካባቢ ሁኔታ መሠረት አተገባበሩ ሊለያይ ይችላል። በትክክል ሲተገበር የምርት ጥፋትን በመቀነስ ገቢን ያሻሽላል እና ተመጣጣኝ አቀራረብ ጠቃሚ ነፍሳትን፣ የውሃ ጥራትን እና
- USER: የተባይ አስተዳደር ለምን አስፈላጊ ነው?
  ASSISTANT: የተባይ አስተዳደር አስፈላጊ የሆነው ተክሎችን ከጎጂ ተባዮች በተመጣጣኝ መንገድ መጠበቅ ስለሚደግፍ ነው። በአካባቢ ሁኔታ መሠረት አተገባበሩ ሊለያይ ይችላል። ይህም በመጨረሻ የምርት ጥፋትን በመቀነስ ገቢን ያሻሽላል እና ተመጣጣኝ አቀራረብ ጠቃሚ ነፍሳትን፣ የውሃ ጥራትን እና የአፈር ሕይ
✅ Data cleaning complete


## SECTION E — Training configuration

T4-safe defaults are used. Do not use A100 batch settings on T4.

In [ ]:
import os, sys, gc, getpass, importlib.metadata as im
from pathlib import Path
from datetime import datetime

import torch
from huggingface_hub import login, whoami, HfApi

print("Python:", sys.version.split()[0])
print("Torch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)

if not torch.cuda.is_available():
    raise RuntimeError("❌ GPU is not available. In Colab: Runtime -> Change runtime type -> T4 GPU.")

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print("GPU:", gpu_name)
print(f"VRAM: {vram_gb:.1f} GB")

# IMPORTANT: use FP16 on T4. T4 is not a real BF16 training GPU.
USE_BF16 = False
USE_FP16 = True
DTYPE = torch.float16
print("Training dtype: FP16")

for pkg in ["transformers", "accelerate", "peft", "datasets"]:
    print(f"{pkg}:", im.version(pkg))

# Fail early if broken packages from older notebook are installed and imported accidentally.
for bad_pkg in ["bitsandbytes", "trl"]:
    try:
        print(f"⚠️ {bad_pkg} is installed, but this notebook will NOT use it:", im.version(bad_pkg))
    except im.PackageNotFoundError:
        print(f"✅ {bad_pkg} not installed / not needed")

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = None

if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Paste Hugging Face READ token (input hidden): ").strip()

if not HF_TOKEN.startswith("hf_"):
    raise ValueError("❌ Invalid Hugging Face token. It should start with hf_")

login(token=HF_TOKEN, add_to_git_credential=False)
info = whoami(token=HF_TOKEN)
print("✅ Hugging Face login successful as:", info.get("name", "unknown"))

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
api = HfApi(token=HF_TOKEN)
api.model_info(MODEL_ID)
print("✅ Qwen access OK:", MODEL_ID)




# SECTION E — Training configuration

from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import AutoTokenizer

SYSTEM_PROMPT_AM = (
    "አንተ ለኢትዮጵያ ግብርና ዘርፍ የተዘጋጀ የአማርኛ AI ረዳት ነህ። "
    "ለገበሬዎች እና ለግብርና ባለሙያዎች ጠቃሚ፣ ትክክለኛ፣ አጭር እና ተግባራዊ መልስ ስጥ። "
    "ሁልጊዜ በአማርኛ ብቻ መልስ። ቻይንኛ፣ እንግሊዝኛ ወይም ሌላ ቋንቋ አትጠቀም።"
)

TRAINING_CONFIG = {
    "BASE_MODEL": MODEL_ID,
    "MAX_SEQ_LEN": 512,       # If OOM on T4, change to 384 or 256.
    "VAL_SPLIT": 0.05,
    "SEED": 42,
    "LORA_R": 8,              # T4-safe. Increase to 16 only on A100.
    "LORA_ALPHA": 16,
    "LORA_DROPOUT": 0.05,
    # ☢️ TO FIX REPETITIVE/GENERIC ANSWERS, CHANGE NUM_EPOCHS TO 3:
    "NUM_EPOCHS": 3,
    "BATCH_SIZE": 1,
    "GRAD_ACCUM": 16,
    "LR": 2e-4,
    "LOG_STEPS": 10,
    "SAVE_STEPS": 200,
    "EVAL_STEPS": 200,
    # ☢️ TO GET THE BEST RESULTS, CHANGE THIS TO None TO USE ALL 52,000 ROWS:
    "MAX_TRAIN_SAMPLES": None
}

if "A100" in gpu_name.upper() or vram_gb >= 35:
    TRAINING_CONFIG.update({
        "LORA_R": 16,
        "LORA_ALPHA": 32,
        "BATCH_SIZE": 2,
        "GRAD_ACCUM": 8,
        "MAX_SEQ_LEN": 768,
    })

print("Current Training Config:")
print(TRAINING_CONFIG)
print("\nℹ️ TIP: For maximum accuracy, ensure NUM_EPOCHS is 3 and MAX_TRAIN_SAMPLES is None (when you have enough compute).")

tokenizer = AutoTokenizer.from_pretrained(
    TRAINING_CONFIG["BASE_MODEL"],
    token=HF_TOKEN,
    trust_remote_code=True,
    padding_side="right",
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
print("✅ Tokenizer loaded:", tokenizer.__class__.__name__)
print("pad_token:", repr(tokenizer.pad_token), tokenizer.pad_token_id)


{'BASE_MODEL': 'Qwen/Qwen2.5-3B-Instruct', 'MAX_SEQ_LEN': 768, 'VAL_SPLIT': 0.05, 'SEED': 42, 'LORA_R': 16, 'LORA_ALPHA': 32, 'LORA_DROPOUT': 0.05, 'NUM_EPOCHS': 1, 'BATCH_SIZE': 2, 'GRAD_ACCUM': 8, 'LR': 0.0002, 'LOG_STEPS': 10, 'SAVE_STEPS': 200, 'EVAL_STEPS': 200, 'MAX_TRAIN_SAMPLES': 25000}
✅ Tokenizer loaded: Qwen2TokenizerFast
pad_token: '<|endoftext|>' 151643


## SECTION F — Tokenize with assistant-only labels

This is important. The model should learn the assistant answer, not waste loss on the system prompt and user question.

In [ ]:
# SECTION F — Build train/validation datasets with assistant-only label masking

import numpy as np, pandas as pd, torch
from datasets import Dataset
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

def make_messages(user_text, assistant_text=None):
    msgs = [
        {"role": "system", "content": SYSTEM_PROMPT_AM},
        {"role": "user", "content": clean_text(user_text)},
    ]
    if assistant_text is not None:
        msgs.append({"role": "assistant", "content": clean_text(assistant_text)})
    return msgs

def tokenize_example(row):
    user_text = row[USER_COL]
    answer_text = row[ASSISTANT_COL]

    prompt_text = tokenizer.apply_chat_template(
        make_messages(user_text, None),
        tokenize=False,
        add_generation_prompt=True,
    )
    full_text = tokenizer.apply_chat_template(
        make_messages(user_text, answer_text),
        tokenize=False,
        add_generation_prompt=False,
    )

    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
    full = tokenizer(
        full_text,
        add_special_tokens=False,
        truncation=True,
        max_length=TRAINING_CONFIG["MAX_SEQ_LEN"],
    )

    input_ids = full["input_ids"]
    attention_mask = full["attention_mask"]
    labels = input_ids.copy()

    # Mask system + user + assistant header. Train only answer tokens.
    prompt_len = min(len(prompt_ids), len(labels))
    labels[:prompt_len] = [-100] * prompt_len

    # If truncation removed the answer, drop this example later.
    supervised_tokens = sum(1 for x in labels if x != -100)
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
        "supervised_tokens": supervised_tokens,
    }

work_df = df.copy()
if TRAINING_CONFIG["MAX_TRAIN_SAMPLES"]:
    work_df = work_df.sample(n=min(TRAINING_CONFIG["MAX_TRAIN_SAMPLES"], len(work_df)), random_state=TRAINING_CONFIG["SEED"]).reset_index(drop=True)

train_df, val_df = train_test_split(
    work_df,
    test_size=TRAINING_CONFIG["VAL_SPLIT"],
    random_state=TRAINING_CONFIG["SEED"],
    shuffle=True,
)

train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
val_dataset = Dataset.from_pandas(val_df.reset_index(drop=True))

remove_cols = list(train_dataset.column_names)
train_dataset = train_dataset.map(tokenize_example, remove_columns=remove_cols, desc="Tokenizing train")
val_dataset = val_dataset.map(tokenize_example, remove_columns=remove_cols, desc="Tokenizing val")

train_dataset = train_dataset.filter(lambda x: x["supervised_tokens"] >= 5, desc="Filtering train")
val_dataset = val_dataset.filter(lambda x: x["supervised_tokens"] >= 5, desc="Filtering val")
train_dataset = train_dataset.remove_columns(["supervised_tokens"])
val_dataset = val_dataset.remove_columns(["supervised_tokens"])

print(f"✅ Train examples: {len(train_dataset):,}")
print(f"✅ Val examples: {len(val_dataset):,}")
print("Example token length:", len(train_dataset[0]["input_ids"]))

Tokenizing train:   0%|          | 0/23750 [00:00<?, ? examples/s]

Tokenizing val:   0%|          | 0/1250 [00:00<?, ? examples/s]

Filtering train:   0%|          | 0/23750 [00:00<?, ? examples/s]

Filtering val:   0%|          | 0/1250 [00:00<?, ? examples/s]

✅ Train examples: 23,750
✅ Val examples: 1,250
Example token length: 521


## SECTION G — Load Qwen2.5 model safely without bitsandbytes

This is the replacement for your failing Qwen loading section.

In [ ]:
# SECTION G — Load Qwen2.5 base model safely WITHOUT bitsandbytes

import gc, torch, os
from transformers import AutoModelForCausalLM

gc.collect()
torch.cuda.empty_cache()

print("Loading base model without bitsandbytes:", TRAINING_CONFIG["BASE_MODEL"])
print("This avoids libnvJitLink.so.13 / Bnb4bitQuantize errors.")

try:
    base_model = AutoModelForCausalLM.from_pretrained(
        TRAINING_CONFIG["BASE_MODEL"],
        token=HF_TOKEN,
        torch_dtype=torch.float16,
        device_map={"": 0},
        low_cpu_mem_usage=True,
        trust_remote_code=True,
        attn_implementation="eager",  # safer on Colab than flash-attn assumptions
    )
except torch.cuda.OutOfMemoryError as e:
    torch.cuda.empty_cache()
    raise RuntimeError(
        "❌ CUDA OOM while loading Qwen2.5-3B in FP16. "
        "Use A100, or change BASE_MODEL to Qwen/Qwen2.5-1.5B-Instruct, or restart runtime and try again."
    ) from e

base_model.config.use_cache = False
try:
    base_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
except TypeError:
    base_model.gradient_checkpointing_enable()

# Freeze base model. LoRA will add trainable weights next.
for p in base_model.parameters():
    p.requires_grad = False

print("✅ Base model loaded")
print("Model class:", base_model.__class__.__name__)
print(f"VRAM allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

Loading base model without bitsandbytes: Qwen/Qwen2.5-3B-Instruct
This avoids libnvJitLink.so.13 / Bnb4bitQuantize errors.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

✅ Base model loaded
Model class: Qwen2ForCausalLM
VRAM allocated: 5.85 GB


## SECTION H — Attach LoRA adapter

LoRA trains only small adapter matrices, not the full Qwen model.

In [ ]:
# SECTION H — Attach LoRA

from peft import LoraConfig, get_peft_model, TaskType

# Qwen2.5 uses these Linear module names.
target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

lora_config = LoraConfig(
    r=TRAINING_CONFIG["LORA_R"],
    lora_alpha=TRAINING_CONFIG["LORA_ALPHA"],
    target_modules=target_modules,
    lora_dropout=TRAINING_CONFIG["LORA_DROPOUT"],
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
if trainable == 0:
    raise RuntimeError("❌ LoRA did not attach. Target modules are wrong.")

print("✅ LoRA attached correctly")
print(f"VRAM allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607
✅ LoRA attached correctly
VRAM allocated: 5.97 GB


## SECTION I — Sanity generation before training

The answer may still be low quality before training. This cell only proves the model can generate without crashing.

In [ ]:
# SECTION I — Sanity generation

import torch
model.eval()

@torch.inference_mode()
def generate_answer(question, max_new_tokens=120, temperature=0.2):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT_AM},
        {"role": "user", "content": question},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=768).to(model.device)
    start_len = inputs["input_ids"].shape[1]
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True if temperature > 0 else False,
        temperature=temperature if temperature > 0 else None,
        top_p=0.9 if temperature > 0 else None,
        repetition_penalty=1.12,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(out[0][start_len:], skip_special_tokens=True).strip()

print(generate_answer("በኢትዮጵያ ለስንዴ ምርት የሚረዱ 3 ምክሮችን ንገረኝ።"))
print("✅ Generation works. Continue to training.")

የኢትዮጵያ ለስንዴ ምርት የሚረዱ 3 ምክር ኢትዮጵያዊ አድmissible የመሰረት የሚገኙው ነው. በአዲስ አበባ, በአንደኛ የነጋገር የሚገኙው የስንዴ ምርት አድmissible የ
✅ Generation works. Continue to training.


## SECTION J — Training with Hugging Face Trainer

No `SFTTrainer`, no `TRL`, no `bitsandbytes`, no `paged_adamw_8bit`. This avoids the common Colab mismatch errors.

In [ ]:
# SECTION J — Train LoRA adapter

import os, inspect, json, torch
from transformers import Trainer, TrainingArguments
from dataclasses import dataclass
from typing import List, Dict, Any

os.environ["WANDB_DISABLED"] = "true"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

@dataclass
class CausalLMCollator:
    tokenizer: any
    label_pad_token_id: int = -100

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        input_ids = [f["input_ids"] for f in features]
        attention_mask = [f["attention_mask"] for f in features]
        labels = [f["labels"] for f in features]
        batch = self.tokenizer.pad(
            {"input_ids": input_ids, "attention_mask": attention_mask},
            padding=True,
            return_tensors="pt",
        )
        max_len = batch["input_ids"].shape[1]
        padded_labels = []
        for lab in labels:
            lab = list(lab)
            lab = lab + [self.label_pad_token_id] * (max_len - len(lab))
            padded_labels.append(lab)
        batch["labels"] = torch.tensor(padded_labels, dtype=torch.long)
        return batch

collator = CausalLMCollator(tokenizer)

# TrainingArguments changed name from evaluation_strategy to eval_strategy in newer versions.
sig = inspect.signature(TrainingArguments.__init__).parameters
eval_key = "eval_strategy" if "eval_strategy" in sig else "evaluation_strategy"

args_dict = dict(
    output_dir=str(CHECKPOINT_DIR),
    num_train_epochs=TRAINING_CONFIG["NUM_EPOCHS"],
    per_device_train_batch_size=TRAINING_CONFIG["BATCH_SIZE"],
    per_device_eval_batch_size=TRAINING_CONFIG["BATCH_SIZE"],
    gradient_accumulation_steps=TRAINING_CONFIG["GRAD_ACCUM"],
    learning_rate=TRAINING_CONFIG["LR"],
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    weight_decay=0.01,
    max_grad_norm=1.0,
    fp16=True,
    bf16=False,
    gradient_checkpointing=True,
    optim="adamw_torch",
    logging_steps=TRAINING_CONFIG["LOG_STEPS"],
    save_strategy="steps",
    save_steps=TRAINING_CONFIG["SAVE_STEPS"],
    save_total_limit=2,
    report_to=[],
    seed=TRAINING_CONFIG["SEED"],
    remove_unused_columns=False,
    dataloader_pin_memory=False,
)
args_dict[eval_key] = "steps"
args_dict["eval_steps"] = TRAINING_CONFIG["EVAL_STEPS"]

# load_best_model_at_end requires matching save/eval strategy and can cause confusion on short runs.
training_args = TrainingArguments(**args_dict)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collator,
    tokenizer=tokenizer,
)

print("Training setup:")
print(" model:", TRAINING_CONFIG["BASE_MODEL"])
print(" train examples:", len(train_dataset))
print(" val examples:", len(val_dataset))
print(" batch:", TRAINING_CONFIG["BATCH_SIZE"], "grad_accum:", TRAINING_CONFIG["GRAD_ACCUM"])
print(" max_seq_len:", TRAINING_CONFIG["MAX_SEQ_LEN"])
print(" checkpoints:", CHECKPOINT_DIR)

try:
    train_result = trainer.train()
except torch.cuda.OutOfMemoryError as e:
    torch.cuda.empty_cache()
    raise RuntimeError(
        "❌ CUDA OOM during training. Restart runtime and change MAX_SEQ_LEN to 384 or 256, "
        "or set MAX_TRAIN_SAMPLES=3000 for a fast deadline run."
    ) from e

print("✅ Training complete")
print(train_result.metrics)

with open(RUN_DIR / "train_metrics.json", "w", encoding="utf-8") as f:
    json.dump({k: float(v) if isinstance(v, (int, float)) else str(v) for k, v in train_result.metrics.items()}, f, indent=2)

You're using a Qwen2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Training setup:
 model: Qwen/Qwen2.5-3B-Instruct
 train examples: 23750
 val examples: 1250
 batch: 2 grad_accum: 8
 max_seq_len: 768
 checkpoints: /content/drive/MyDrive/amharic_agri_ai/final_lora_no_bnb_20260504_1902/checkpoints


Step,Training Loss,Validation Loss
200,0.037800,0.031455
400,0.010600,0.012968
600,0.007900,0.008696
800,0.006200,0.007351
1000,0.005800,0.006023
1200,0.005100,0.005463
1400,0.005300,0.005189


✅ Training complete
{'train_runtime': 6618.1481, 'train_samples_per_second': 3.589, 'train_steps_per_second': 0.224, 'total_flos': 2.4612602577985536e+17, 'train_loss': 0.054074649624978356, 'epoch': 0.9997473684210526}


## SECTION K — Save adapter

In [ ]:
# SECTION K — Save LoRA adapter and tokenizer

import json, shutil, os

trainer.save_model(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))

with open(ADAPTER_DIR / "training_config.json", "w", encoding="utf-8") as f:
    json.dump(TRAINING_CONFIG, f, ensure_ascii=False, indent=2)

print("✅ Adapter saved:", ADAPTER_DIR)
print("Files:")
for p in sorted(ADAPTER_DIR.glob("*")):
    if p.is_file():
        print("-", p.name, f"{p.stat().st_size/1024/1024:.2f} MB")

✅ Adapter saved: /content/drive/MyDrive/amharic_agri_ai/final_lora_no_bnb_20260504_1902/lora_adapter
Files:
- README.md 0.00 MB
- adapter_config.json 0.00 MB
- adapter_model.safetensors 114.25 MB
- added_tokens.json 0.00 MB
- merges.txt 1.59 MB
- special_tokens_map.json 0.00 MB
- tokenizer.json 10.89 MB
- tokenizer_config.json 0.01 MB
- training_args.bin 0.01 MB
- training_config.json 0.00 MB
- vocab.json 2.65 MB


## SECTION L — Test trained model

In [ ]:
# SECTION L — Inference test after training

model.eval()
TEST_QUESTIONS = [
    "ቡና ለማምረት ምን ዓይነት አፈር ያስፈልጋል?",
    "ድርቅ ሲኖር ሰብሌን እንዴት ማዳን እችላለሁ?",
    "ለስንዴ ማዳበሪያ መቼ መስጠት ይሻላል?",
    "የበቆሎ ቢጫ ቅጠል ምን ምልክት ያሳያል?",
    "የተባይ አስተዳደር በምርት ሂደት ውስጥ ምን ሚና ይጫወታል?"
    "የበሽታ ቁጥጥር ውጤታማ መሆኑን እንዴት እንለካ?"

]

results = []
for q in TEST_QUESTIONS:
    ans = generate_answer(q, max_new_tokens=220, temperature=0.2)
    results.append({"question": q, "answer": ans})
    print("\nQ:", q)
    print("A:", ans)

print("\n" + "="*60)
print("⚠️ NOTE ON QUALITY / UNDERFITTING:")
print("If answers seem repetitive or don't directly answer the question")
print("(e.g., constantly repeating 'በአካባቢ ሁኔታ መሠረት...'), this is expected.")
print("Because of compute limits, this model trained for only 1 Epoch on half the data.")
print("To get highly accurate, specific agricultural answers, restart the notebook")
print("and change NUM_EPOCHS=3 and MAX_TRAIN_SAMPLES=None in Section E.")
print("="*60 + "\n")

pd.DataFrame(results).to_csv(REPORT_DIR / "inference_results.csv", index=False)
print("✅ Saved inference results:", REPORT_DIR / "inference_results.csv")

NameError: name 'model' is not defined

## SECTION M — Simple evaluation and final report

In [ ]:
# SECTION M — Evaluation report

import json, re, pandas as pd

def amharic_percent(text):
    text = str(text)
    chars = re.findall(r"[\u1200-\u137F\u0041-\u007A\u0600-\u06FF\u4E00-\u9FFF]", text)
    amh = re.findall(r"[\u1200-\u137F]", text)
    return round(100 * len(amh) / max(len(chars), 1), 1)

AGRI_KEYWORDS = ["አፈር", "ማዳበሪያ", "ውሃ", "ሰብል", "ምርት", "ዘር", "ግብርና", "ቡና", "ስንዴ", "በቆሎ", "ድርቅ"]

def agri_score(text):
    return round(100 * sum(1 for k in AGRI_KEYWORDS if k in str(text)) / len(AGRI_KEYWORDS), 1)

eval_rows = []
for r in results:
    eval_rows.append({
        "question": r["question"],
        "answer": r["answer"],
        "amharic_percent": amharic_percent(r["answer"]),
        "agri_keyword_percent": agri_score(r["answer"]),
        "answer_chars": len(r["answer"]),
    })

eval_df = pd.DataFrame(eval_rows)
eval_df.to_csv(REPORT_DIR / "simple_eval.csv", index=False)

report = {
    "project": "Amharic Ethiopian Agriculture Qwen2.5 LoRA fine-tuning",
    "base_model": TRAINING_CONFIG["BASE_MODEL"],
    "approach": "FP16 LoRA without bitsandbytes, TRL, or SFTTrainer",
    "reason_for_change": "Avoid Colab CUDA 13 / bitsandbytes libnvJitLink.so.13 failure",
    "clean_rows": len(df),
    "train_examples": len(train_dataset),
    "val_examples": len(val_dataset),
    "adapter_dir": str(ADAPTER_DIR),
    "avg_amharic_percent": float(eval_df["amharic_percent"].mean()),
    "avg_agri_keyword_percent": float(eval_df["agri_keyword_percent"].mean()),
    "training_config": TRAINING_CONFIG,
}

with open(REPORT_DIR / "final_report.json", "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print("✅ Final report saved:", REPORT_DIR / "final_report.json")
print(json.dumps(report, ensure_ascii=False, indent=2))

NameError: name 'results' is not defined

## Emergency settings if T4 still runs out of memory

Change these in Section E and restart from Section E:

```python
TRAINING_CONFIG["MAX_SEQ_LEN"] = 384   # or 256
TRAINING_CONFIG["MAX_TRAIN_SAMPLES"] = 3000
TRAINING_CONFIG["LORA_R"] = 4
TRAINING_CONFIG["LORA_ALPHA"] = 8
```

If the 3B model cannot load in FP16 on your assigned T4 session, change:

```python
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
```

But try the 3B notebook first.